# Setup

In [1]:
import json
import os
import sys
from datetime import datetime, timedelta
import hashlib
import hmac
import time

import requests
from dotenv import load_dotenv

## Constants

In [30]:
base_url = "https://open-api.tiktokglobalshop.com"

In [20]:
load_dotenv()  # reads .env in the current directory into environment variables

app_key = os.environ.get("TIKTOK_APP_KEY")
app_secret = os.environ.get("TIKTOK_APP_SECRET")
auth_code = os.environ.get("TIKTOK_AUTH_CODE")
shop_cipher = os.environ.get("SHOP_CIPHER")

## Helper functions

In [18]:
def generate_sign(path: str, params: dict, app_secret: str, body: str = "") -> str:
    """
    Generate the TikTok Shop 'sign' value for a request.

    Args:
        path: The request path only, e.g. "/authorization/202309/shops"
              (no domain, no query string).
        params: Dict of query parameters that will be sent with the request.
                Include app_key and timestamp here. Do NOT include 'sign' itself
                (it's excluded automatically even if present).
        app_secret: Your TikTok Shop App Secret.
        body: Raw JSON request body string, only for requests that have one
              (e.g. POST with a JSON payload). Leave as "" for GET requests
              or requests with no body.

    Returns:
        Lowercase hex-encoded HMAC-SHA256 signature string.
    """
    # Step 1: sort params alphabetically, excluding sign/access_token
    filtered = {k: v for k, v in params.items() if k not in ("sign", "access_token")}
    sorted_params = sorted(filtered.items())

    # Step 2: concatenate sorted param names + values
    param_string = "".join(f"{k}{v}" for k, v in sorted_params)

    # Step 3: prepend the path
    base_string = f"{path}{param_string}"

    # Step 4: append the raw body, if any
    if body:
        base_string += body

    # Step 5: wrap with app_secret on both ends, then HMAC-SHA256
    signed_string = f"{app_secret}{base_string}{app_secret}"

    return hmac.new(
        app_secret.encode("utf-8"),
        signed_string.encode("utf-8"),
        hashlib.sha256,
    ).hexdigest()

In [19]:
def build_signed_params(path: str, params: dict, app_secret: str, body: str = "") -> dict:
    """
    Convenience wrapper: returns params with 'sign' added, ready to send.
    Does not mutate the input dict.
    """
    params_with_sign = dict(params)
    params_with_sign["sign"] = generate_sign(path, params, app_secret, body)
    return params_with_sign

# API Setup

## 1. Get Access Token

In [29]:
token_url = "https://auth.tiktok-shops.com/api/v2/token/get"

In [4]:
params = {
    "app_key": app_key,
    "app_secret": app_secret,
    "auth_code": auth_code,
    "grant_type": "authorized_code",
}

In [5]:
response = requests.get(token_url, params=params, timeout=15)
response.raise_for_status()

In [7]:
response.json()

{'code': 0,
 'message': 'success',
 'data': {'access_token': 'ROW_8GLOGwAAAAC-aSSBzs9hTkAvQpUHc-NId6w78Jeod0Gn65cgjrPqtgwhAeWpFp2dr7Y0Odhz0AS07Xl7BKOrKt84qUPr4bX4XoZ7OygNVq5bC7spz-tgjQ',
  'access_token_expire_in': 1784431523,
  'refresh_token': 'ROW_I6mkXwAAAAD0azg6QUtmdHanP9Amo6ED-HvHHWt26D7McAx-0QGawHBFkzxria_YOll6NlShG54',
  'refresh_token_expire_in': 4905887715,
  'open_id': 'Qc7EdAAAAACBUux6vr1HusjIeqDvzh9Px2CAz8oXZSLBZpSfkx-amw',
  'seller_name': 'Vitami',
  'seller_base_region': 'PH',
  'user_type': 0,
  'granted_scopes': ['seller.authorization.info']},
 'request_id': '20260712112523585CF541F021554174AA'}

In [8]:
access_token = response.json()['data']['access_token']

## 2. Get Shop Cipher

In [25]:
path = "/authorization/202309/shops"

params = {
    "app_key": app_key,
    "timestamp": int(time.time()),
}
signed_params = build_signed_params(path, params, app_secret)

headers = {
    "x-tts-access-token": access_token,
    "content-type": "application/json",
}

response = requests.get(f"{base_url}{path}", params=signed_params, headers=headers, timeout=15)
print(f"Request URL: {response.url}")
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")

Request URL: https://open-api.tiktokglobalshop.com/authorization/202309/shops?app_key=6kjiaja22gomi&timestamp=1783827261&sign=f0385c2f082372773872eba9ad6afc105fa4406e0748a3c3b5c2042f42b9fabf
Status: 200
Response: {'code': 0, 'data': {'shops': [{'cipher': 'ROW_N6cpKQAAAADl57_xyhcxuGLS8aCEeYMb', 'code': 'PHPHLCVULTUD', 'id': '7494459291006306136', 'name': 'Vitami', 'region': 'PH', 'seller_type': 'LOCAL'}]}, 'message': 'Success', 'request_id': '202607121134215D3388B507EB6C667685'}


In [15]:
response.json()['data']['shops'][0]

{'cipher': 'ROW_N6cpKQAAAADl57_xyhcxuGLS8aCEeYMb',
 'code': 'PHPHLCVULTUD',
 'id': '7494459291006306136',
 'name': 'Vitami',
 'region': 'PH',
 'seller_type': 'LOCAL'}

In [21]:
shop_cipher

'ROW_N6cpKQAAAADl57_xyhcxuGLS8aCEeYMb'

# API Calls

## Test Affiliate API access

In [31]:
path = "/affiliate_seller/202409/open_collaboration_settings"
params = {
    "app_key": app_key,
    "timestamp": int(time.time()),
    "shop_cipher": shop_cipher,
}

signed_params = build_signed_params(path, params, app_secret)

response = requests.get(
    f"{base_url}{path}",
    params=signed_params,
    headers={"x-tts-access-token": access_token, "content-type": "application/json"},
    timeout=15,
)
print(f"Request URL: {response.url}")
print(f"Status: {response.status_code}")

Request URL: https://open-api.tiktokglobalshop.com/affiliate_seller/202409/open_collaboration_settings?app_key=6kjiaja22gomi&timestamp=1783827318&shop_cipher=ROW_N6cpKQAAAADl57_xyhcxuGLS8aCEeYMb&sign=4052209ccbfbd72219b20cdb0448826e3ee907b341641809e2d86b282ab62d06
Status: 401


In [32]:
response.json()

{'code': 105005,
 'message': 'Access denied. The app is not authorized to access the endpoint because the access scopes granted for the app or the access token do not contain the required access scope for the endpoint. For more details: https://m.tiktok.shop/s/AIu6dbFhs2XW',
 'request_id': '20260712113518E9CC511F21E49D43B5A2'}